# RNA QC + 细胞类型注解（薄 notebook：调参 + 可视化）

**全部计算在 `core/lemmus/downstream/qc.py --analyze` 中完成**（仿照 Alopex 的 `alopex.downstream.qc` 模式）：QC 表、scanpy 流程（过滤/HVG/PCA/Leiden/UMAP/marker）、物种分支注解（human=Census scVI consensus；mouse=deterministic marker program）和 RNA×DNAme 整合都由 Processor 子进程产出，写入 `06_downstream/`。本 notebook 只负责：调参、启动 Processor、读取结果并画图。

**物种分支**：human（hg38/GRCh38）走 Census scVI consensus 主注解；小鼠（mm10/GRCm38/GRCm39）走 deterministic cluster-level marker-program 注解（Census 只发布 human pretrained scVI model），字段与导出合同两物种一致。

In [ ]:
from pathlib import Path
from importlib.metadata import version
import os
import sys
import json

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import anndata
import scanpy as sc

# ---- 内核环境自检：选错内核时给出可操作的报错，而不是中途神秘失败 ----
import importlib

_MISSING = [m for m in ("scanpy", "anndata", "igraph", "leidenalg", "seaborn", "yaml")
            if importlib.util.find_spec(m) is None]
if _MISSING:
    raise RuntimeError(
        f"当前内核缺少 notebook 依赖: {_MISSING}。\n"
        "请在 Jupyter/编辑器中选择内核 \"RNA DARLIN Atlas (managed)\"（对应 "
        "/Users/felix/Documents/RNA_DARLIN_Pipeline/conda/current/atlas），"
        "并重新 Reload notebook 后从头运行。"
    )

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(context='paper', style='ticks')
mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#4A4A4A', 'axes.linewidth': 0.75,
    'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'font.size': 9,
    'legend.fontsize': 8, 'legend.title_fontsize': 8,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor='white', frameon=False, vector_friendly=True, fontsize=9)

print('scanpy', version('scanpy'), '| anndata', version('anndata'), '| kernel:', sys.executable)

## 0. 调参与启动 Processor

计算参数（过滤阈值、HVG/PCA/Leiden/UMAP）写入 `ANALYSIS_OVERRIDES`，仅覆盖需要改动的项，其余用 Processor 缺省值；DNAme 整合由 `DNA_PROJECT_ROOT` 启用（在 Processor 内完成，包括 `dna_raw_sample -> rna_sample` 的 CloneID 桥接）。

In [ ]:
# ============ 用户可调参数（运行前先确认） ============
# Pipeline 根目录解析：格内显式值 > 环境变量 > 项目同级目录扫描（与 DNA_QC_Visualization 同语义）。
RNA_PROJECT_ROOT = None    # 例如 Path("/absolute/path/to/RNA_DARLIN_project")
RNA_PIPELINE_ROOT = None   # 例如 Path("/absolute/path/to/RNA_DARLIN_Pipeline")
DNA_PROJECT_ROOT = None  # 可选：例如 Path("/absolute/path/to/DNA_project")，启用 DNAme 整合

# ---- 细胞 QC 阈值（HQ 细胞标准；本格集中可调，修改后从本格起顺序重跑）----
# UMI／基因数两条阈值共同决定 qc_pass；MT% 仅用于诊断；小提琴图会画出实际生效值（红线），被滤细胞以 × 标注。
# min_genes_per_cell: 每细胞最低 supported 基因数（>=1.0 expected UMI 计为检出），Processor 缺省 300
# min_counts_per_cell: 每细胞最低 UMI 计数，Processor 缺省 1000
QC_THRESHOLDS = {
    "min_genes_per_cell": 300,
    "min_counts_per_cell": 1000,
}

# 其他分析参数覆盖（Processor 缺省见 lemmus.downstream.config.DEFAULT_ANALYSIS_PARAMS）：
# min_cells_per_gene / n_top_hvg / n_pcs / n_neighbors / leiden_resolution /
# umap_min_dist / random_state
ANALYSIS_OVERRIDES = {**QC_THRESHOLDS}

from pathlib import Path
import os

def _validate_pipeline_root(candidate) -> Path:
    candidate = Path(candidate).expanduser().resolve()
    if not (candidate / "core/lemmus/downstream/qc.py").is_file():
        raise FileNotFoundError(f"Not a usable Lemmus: {candidate}")
    return candidate

def _configured_project_root() -> Path:
    option = RNA_PROJECT_ROOT or os.environ.get("RNA_PROJECT_ROOT")
    if option:
        candidate = Path(option).expanduser().resolve()
        if not (candidate / "03_results" / "count").is_dir():
            raise FileNotFoundError(f"RNA_PROJECT_ROOT 缺少 03_results/count: {candidate}")
        return candidate
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / "03_results" / "count").is_dir():
            return base.resolve()
    raise FileNotFoundError("Cannot locate RNA project (03_results/count); set RNA_PROJECT_ROOT in this cell.")

def _configured_pipeline_root(project_dir: Path) -> Path:
    option = RNA_PIPELINE_ROOT or os.environ.get("RNA_PIPELINE_ROOT")
    if option:
        return _validate_pipeline_root(Path(option))
    siblings = sorted(
        sibling for sibling in project_dir.parent.iterdir()
        if sibling.is_dir() and (sibling / "core/lemmus/downstream/qc.py").is_file()
    )
    if len(siblings) == 1:
        return _validate_pipeline_root(siblings[0])
    if len(siblings) > 1:
        raise FileNotFoundError(
            "多个同级目录都包含 core/lemmus/downstream/qc.py: "
            + ", ".join(s.name for s in siblings) + "；请显式填写 RNA_PIPELINE_ROOT。"
        )
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / "core/lemmus/downstream/qc.py").is_file():
            return _validate_pipeline_root(base)
    raise FileNotFoundError(f"Cannot find Lemmus for project {project_dir}; set RNA_PIPELINE_ROOT.")

PROJECT_DIR = _configured_project_root()
PIPELINE_PATH = _configured_pipeline_root(PROJECT_DIR)
print("PROJECT_DIR      =", PROJECT_DIR)
print("PIPELINE_PATH    =", PIPELINE_PATH)
print("DNA_PROJECT_ROOT =", DNA_PROJECT_ROOT)
print("ANALYSIS_OVERRIDES =", ANALYSIS_OVERRIDES)

In [ ]:
# ---- 以子进程运行 Processor（全部计算在此完成；本内核只负责可视化）----
import subprocess

PROCESSOR_MODULE = "lemmus.downstream.qc"
CORE_MODULE_DIR = PIPELINE_PATH / "core"
if str(CORE_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(CORE_MODULE_DIR))
from lemmus.downstream import qc as _qc_module
if Path(_qc_module.__file__).resolve() != (PIPELINE_PATH / "core/lemmus/downstream/qc.py").resolve():
    raise RuntimeError("内核已导入另一份 Lemmus，请重启内核。")
from lemmus.downstream.config import RNA_QC_COLUMNS
print(f"INFO: RNA 分析开始；解释器：{sys.executable}")

_FLAG_BY_KEY = {
    "min_genes_per_cell": "--min-genes-per-cell",
    "min_counts_per_cell": "--min-counts-per-cell",
    "min_cells_per_gene": "--min-cells-per-gene",
    "n_top_hvg": "--n-top-hvg",
    "n_pcs": "--n-pcs",
    "n_neighbors": "--n-neighbors",
    "leiden_resolution": "--leiden-resolution",
    "umap_min_dist": "--umap-min-dist",
    "random_state": "--random-state",
    "exclude_ig_from_pca": "--keep-ig-in-pca",
    "exclude_mt_from_pca": "--keep-mt-in-pca",
}
cmd = [sys.executable, "-m", PROCESSOR_MODULE, "--project-dir", str(PROJECT_DIR), "--analyze"]
for key, value in ANALYSIS_OVERRIDES.items():
    if key not in _FLAG_BY_KEY:
        raise KeyError(f"未知分析参数: {key}; 可用: {sorted(_FLAG_BY_KEY)}")
    flag = _FLAG_BY_KEY[key]
    if flag.startswith("--keep-"):
        if value is False:
            cmd.append(flag)
    else:
        cmd.extend([flag, str(value)])
if DNA_PROJECT_ROOT is not None:
    cmd.extend(["--dna-project-root", str(DNA_PROJECT_ROOT)])

_processor_env = {**os.environ, "PYTHONPATH": str(PIPELINE_PATH / "core")}
completed = subprocess.run(cmd, check=False, env=_processor_env)
if completed.returncode:
    raise RuntimeError(f"RNA downstream Processor failed with exit code {completed.returncode}")



QC_RESULTS_DIR = PROJECT_DIR / "06_downstream"
FIGURES_DIR = QC_RESULTS_DIR / "plots"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(fig, stem: str):
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")

def clean_axes(ax, grid_y=False):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.75)
    ax.spines['bottom'].set_linewidth(0.75)
    if grid_y:
        ax.grid(axis='y', color='#DADADA', linewidth=0.55, alpha=0.55)

ANALYSIS_SUMMARY = json.loads((QC_RESULTS_DIR / "analysis_summary.json").read_text(encoding="utf-8"))
print("QC_RESULTS_DIR =", QC_RESULTS_DIR)
print("species:", ANALYSIS_SUMMARY["species"],
      "| cells:", ANALYSIS_SUMMARY["n_cells"],
      "| total cells (published H5AD):", ANALYSIS_SUMMARY.get("n_cells_total"),
      "| annotation:", ANALYSIS_SUMMARY["annotation"])
_qp = ANALYSIS_SUMMARY["params"]
print(f"生效 QC 阈值 (analysis_summary.json): min_genes_per_cell={_qp['min_genes_per_cell']}, "
      f"min_counts_per_cell={_qp['min_counts_per_cell']}；MT% 仅诊断，不参与剔除")

## 1. RNA / DARLIN 每细胞 QC 表（13 列发布 schema：含 `pct_counts_mt` 与 4 个注释列）

`pct_counts_mt` 由 Processor 从计数矩阵计算（`gene_symbol` 的 `MT-`/`mt-` 匹配，大小写不敏感），仅用于诊断，不参与 HQ 细胞剔除；原始线粒体基因（含 MT-RNR1/MT-RNR2）及计数保留。全细胞口径的过滤指标与判定见 `RNA_cell_qc_metrics.csv`。

In [ ]:
from IPython.display import display

QC_INFO_PATH = QC_RESULTS_DIR / "RNA_QC_Information.csv"
if not QC_INFO_PATH.is_file():
    raise FileNotFoundError(f"找不到 QC 数据文件: {QC_INFO_PATH}；请先成功运行上一格 Processor。")

df_qc = pd.read_csv(QC_INFO_PATH, dtype={"Sample_ID": str})
if list(df_qc.columns) != list(RNA_QC_COLUMNS):
    raise ValueError(f"RNA_QC_Information.csv 列不符合发布 schema: {list(df_qc.columns)}")
print(f"✅ 已加载 QC 数据: {len(df_qc)} 个细胞, {df_qc['CloneID'].nunique()} 个 sample")
display(df_qc.head())

print("RNA 技术指标概览")
display(df_qc.describe(include=[float, int]).T.round(3))

print("DARLIN clone 检出汇总（详细 QC 证据见 03_results/DARLIN/DARLIN_clone_id/DARLIN_Clone_ID.csv）")
display(df_qc["DARLIN_Clone_ID"].dropna().value_counts().to_frame("n_cells").head(20))
print("按 sample 的 DARLIN 检出率")
display(
    df_qc.assign(has_clone=df_qc["DARLIN_Clone_ID"].notna())
    .groupby("CloneID")["has_clone"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "n_with_clone", "count": "n_cells", "mean": "detect_rate"})
    .round(3)
)

In [ ]:
# QC 总览（13 列发布 schema；保存到 06_downstream/plots/）
pct_mt = pd.to_numeric(df_qc["pct_counts_mt"], errors="coerce")
fig, axes = plt.subplots(2, 3, figsize=(13.0, 8.0))

sns.histplot(df_qc["mapping_rate"].astype(float), bins=30, ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set_title("RNA mapping rate"); axes[0, 0].set_xlabel("mapped reads fraction")

sns.histplot(df_qc["dedup_rate"].astype(float), bins=30, ax=axes[0, 1], color="#5E9C76")
axes[0, 1].set_title("RNA dedup rate"); axes[0, 1].set_xlabel("deduplicated reads fraction")

genes = pd.to_numeric(df_qc["num_genes_expressed"], errors="coerce")
over_mean = pd.to_numeric(df_qc["num_genes_over_mean"], errors="coerce")
sns.scatterplot(x=genes, y=over_mean, s=18, alpha=0.6, linewidth=0, ax=axes[0, 2], color="#B08A3E")
axes[0, 2].set_title("Gene complexity")
axes[0, 2].set_xlabel("genes expressed"); axes[0, 2].set_ylabel("genes over library mean")

sns.histplot(pct_mt, bins=30, ax=axes[1, 0], color="#B25B5B")
axes[1, 0].set_title("Mitochondrial counts (diagnostic only)")
axes[1, 0].set_xlabel("pct_counts_mt (%)")

clone_calls = df_qc["DARLIN_Clone_ID"].dropna()
if len(clone_calls):
    order = clone_calls.value_counts().index
    sns.countplot(x=clone_calls, order=order, ax=axes[1, 1], color="#7567A7")
    axes[1, 1].set_title("DARLIN clone detection")
    axes[1, 1].set_xlabel(""); axes[1, 1].set_ylabel("cells")
    axes[1, 1].tick_params(axis="x", rotation=35)
else:
    axes[1, 1].set_axis_off(); axes[1, 1].set_title("DARLIN clone (无检出)")
axes[1, 2].set_axis_off()

fig.tight_layout()
save_figure(fig, "01_RNA_DARLIN_QC_overview")
plt.show()

## 0.1 DNAme 整合表（可选，由 Processor 生成）

两侧 library 命名不同时，Processor 按 DNA 项目 `00_config/sample_manifest.tsv` 的 `dna_raw_sample -> rna_sample` 对应关系桥接 CloneID 后再按 `(CloneID, PlateID)` 对齐。

In [ ]:
INTEGRATED_PATH = QC_RESULTS_DIR / "Integrated_QC_Information.csv"
if INTEGRATED_PATH.is_file():
    df_integrated = pd.read_csv(INTEGRATED_PATH, dtype={"Sample_ID": str})
    print(f"✅ 整合 QC 表: {df_integrated.shape[0]} 个细胞, {df_integrated.shape[1]} 列 -> {INTEGRATED_PATH}")
    display(df_integrated.head(3))
else:
    df_integrated = None
    print("未生成 Integrated_QC_Information.csv（Processor 未提供 --dna-project-root 或整合失败）。")

## 2. 加载 Processor 注释结果

读取 `RNA_*_annotated.h5ad`（含 UMAP、注释列与 uns 溯源）、Leiden top-30 marker 表和 analysis_summary。

In [ ]:
ANNOTATED_H5AD = QC_RESULTS_DIR / ANALYSIS_SUMMARY["annotated_h5ad"]
adata = sc.read_h5ad(ANNOTATED_H5AD)
markers_top30 = pd.read_csv(QC_RESULTS_DIR / "RNA_Leiden_top30_markers.csv")
print(adata)
print("\n细胞类型主标签（cluster-level broad cell_type_final）:")
print(adata.obs["cell_type_final"].value_counts().to_string())
print("\nDetail 标签层级:")
print(pd.crosstab(adata.obs["cell_type_detail_final"], adata.obs["cell_type_detail_level"]).to_string())

In [ ]:
# RNA QC 小提琴（按 library；全细胞口径 + QC 阈值线）
# 数据源 RNA_cell_qc_metrics.csv：Processor 在过滤前写出，包含被 QC 过滤的细胞；
# 阈值取自 analysis_summary.json 的实际运行参数。本格只做可视化，不做计算。
CELL_QC_PATH = QC_RESULTS_DIR / "RNA_cell_qc_metrics.csv"
if not CELL_QC_PATH.is_file():
    raise FileNotFoundError(f"找不到 {CELL_QC_PATH}；请先成功运行上一格 Processor。")
df_cell_qc = pd.read_csv(CELL_QC_PATH, dtype={"Sample_ID": str})
qc_params = ANALYSIS_SUMMARY["params"]
n_failed = int((~df_cell_qc["qc_pass"].astype(bool)).sum())
print(f"全细胞 {len(df_cell_qc)}；qc_pass {len(df_cell_qc) - n_failed} / 被过滤 {n_failed}（红线 = QC 阈值，× = 被该阈值移除的细胞）")

# library 列转 category 并导出 library_categories（后续 UMAP 格复用同一变量契约）
if not isinstance(adata.obs['library'].dtype, pd.CategoricalDtype):
    adata.obs['library'] = adata.obs['library'].astype('category')
library_categories = list(adata.obs['library'].cat.categories)
library_order = library_categories
library_qc_palette = {
    lib: mpl.colors.to_hex(plt.get_cmap('tab20')(i % 20))
    for i, lib in enumerate(library_order)
}

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1))
for ax, y, title, ylabel, threshold, gate_label, gate_fail in [
    (axes[0], 'n_genes_supported', 'Supported genes', 'Genes with >=1 expected UMI / cell',
     qc_params['min_genes_per_cell'], f"QC: >= {qc_params['min_genes_per_cell']:g} genes", 'below'),
    (axes[1], 'total_counts', 'UMI counts', 'UMIs / cell',
     qc_params['min_counts_per_cell'], f"QC: >= {qc_params['min_counts_per_cell']:g} UMIs", 'below'),
    (axes[2], 'pct_counts_mt', 'Mitochondrial counts (diagnostic)', 'Mitochondrial counts (%)',
     None, None, None),
]:
    sns.violinplot(data=df_cell_qc, x='library', y=y, hue='library', palette=library_qc_palette,
                   order=library_order, legend=False, inner=None, cut=0, linewidth=0.7,
                   saturation=0.88, ax=ax)
    sns.boxplot(data=df_cell_qc, x='library', y=y, order=library_order, width=0.16, showfliers=False,
                boxprops={'facecolor':'white','edgecolor':'#4A4A4A','linewidth':0.65},
                whiskerprops={'color':'#4A4A4A','linewidth':0.65},
                capprops={'color':'#4A4A4A','linewidth':0.65},
                medianprops={'color':'#303030','linewidth':0.9}, ax=ax)
    if threshold is not None:
        ax.axhline(threshold, color='#C0392B', linestyle='--', linewidth=1.1, label=gate_label)
        failed_gate = df_cell_qc.loc[(~df_cell_qc['qc_pass'].astype(bool))
                                 & (df_cell_qc[y] < threshold if gate_fail == 'below'
                                    else df_cell_qc[y] >= threshold)]
        if len(failed_gate):
            ax.scatter(failed_gate['library'], failed_gate[y], marker='x', s=34,
                   color='#C0392B', linewidths=1.4, zorder=5,
                   label=f'removed by this gate ({len(failed_gate)})')
    ax.set_title(title, pad=8); ax.set_ylabel(ylabel); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
    clean_axes(ax, grid_y=True)
    if threshold is not None:
        ax.legend(loc='upper right', fontsize=7.5, frameon=False)
fig.tight_layout(w_pad=1.8)
save_figure(fig, '01_RNA_QC_by_library')
plt.show()

In [ ]:
# 主 UMAP 总览
leiden_categories = list(adata.obs['leiden'].cat.categories)
leiden_umap_palette = [mpl.colors.to_hex(plt.get_cmap('tab10')(i % 10)) for i in range(len(leiden_categories))]
adata.uns['leiden_colors'] = leiden_umap_palette
library_umap_palette = {
    lib: mpl.colors.to_hex(plt.get_cmap('tab20')(i % 20))
    for i, lib in enumerate(library_categories)
}
UMAP_POINT_SIZE = 70

fig, axes = plt.subplots(2, 2, figsize=(9.6, 8.0))
sc.pl.umap(adata, color='leiden', palette=leiden_umap_palette, ax=axes[0, 0], show=False,
           legend_loc='on data', legend_fontsize=8, size=UMAP_POINT_SIZE, alpha=0.95,
           title='RNA Leiden', frameon=False)
sc.pl.umap(adata, color='library', palette=library_umap_palette, ax=axes[0, 1], show=False,
           legend_loc='right margin', size=UMAP_POINT_SIZE, alpha=0.95, title='Library', frameon=False)
sc.pl.umap(adata, color='n_genes_supported', cmap='viridis', ax=axes[1, 0], show=False,
           size=UMAP_POINT_SIZE, alpha=0.95, title='RNA complexity', frameon=False)
sc.pl.umap(adata, color='total_counts', cmap='viridis', ax=axes[1, 1], show=False,
           size=UMAP_POINT_SIZE, alpha=0.95, title='UMI counts', frameon=False)
fig.tight_layout()
save_figure(fig, '02_RNA_UMAP_QC_overview')
plt.show()

In [ ]:
# Leiden unbiased marker 基因（Processor 已计算，这里展示 + 表达定位）
display(markers_top30.groupby("leiden", observed=True).head(10))

top_marker_genes = [
    markers_top30.loc[markers_top30["leiden"] == str(cluster), "names"].iloc[0]
    for cluster in adata.obs["leiden"].cat.categories
    if (markers_top30["leiden"] == str(cluster)).any()
]
top_marker_genes = [g for g in dict.fromkeys(top_marker_genes) if g in adata.var["gene_symbol"].values][:12]
if top_marker_genes:
    sc.pl.dotplot(adata, top_marker_genes, groupby="leiden", standard_scale="var", return_fig=False)
    plt.title("Top marker per Leiden cluster (gene_symbol)")
    save_figure(plt.gcf(), "03_marker_dotplot")
    plt.show()

## 3. 注释放览（物种分支）

human：Census scVI consensus（4 classifier + Cell Ontology 聚合）；mouse：deterministic marker program（支持性检出 ≥1.0 expected UMI + marker gate）。两种注解都由 Processor 在 cluster 级决策并写入同一套字段。

In [ ]:
SPECIES = ANALYSIS_SUMMARY["species"]

if SPECIES == "mouse":
    cluster_review = pd.read_csv(QC_RESULTS_DIR / "RNA_mouse_cluster_marker_review.csv", index_col="cluster")
    print("=== Cluster-level marker-program annotation ===")
    display(cluster_review[[
        "n_cells", "cell_type_final", "cell_type_final_confidence",
        "cell_type_detail_final", "cell_type_detail_level", "cell_type_final_reason",
    ]])
    print("\n=== Lineage detection scores (mean fraction expr > 0, >=1 expected UMI) ===")
    display(cluster_review.filter(regex=r"^detection_"))
    print("\n=== Gated marker support (adjusted P <= 0.05, logFC >= 1) ===")
    display(cluster_review.filter(regex=r"^gated_"))
else:
    cluster_review = pd.read_csv(QC_RESULTS_DIR / "RNA_cluster_annotation_review.csv")
    print("=== Census scVI consensus cluster review ===")
    display(cluster_review.head(20))

print("\n=== Cluster x library audit ===")
display(pd.read_csv(QC_RESULTS_DIR / "RNA_cluster_by_library.csv", index_col=0))
print("=== Broad final label x library audit ===")
display(pd.read_csv(QC_RESULTS_DIR / "RNA_broad_annotation_by_library.csv", index_col=0))
print("=== Sample tissue metadata ===")
display(pd.read_csv(QC_RESULTS_DIR / "RNA_sample_tissue_summary.csv"))

In [ ]:
# 注释 + 组织 UMAP
if SPECIES == "mouse":
    fig, axes = plt.subplots(2, 2, figsize=(13.6, 10.6))
    panels = [
        ("leiden", "Leiden cluster", "on data"),
        ("cell_type_final", "Mouse marker-program broad annotation", "right margin"),
        ("cell_type_final_confidence", "Broad annotation confidence", "right margin"),
        ("cell_type_detail_final", "Detail annotation (B/plasma state; else broad)", "right margin"),
    ]
    for ax, (color, title, legend) in zip(axes.ravel(), panels):
        sc.pl.umap(adata, color=color, ax=ax, show=False, title=title, legend_loc=legend, frameon=False)
    for ax in axes.ravel():
        ax.set_xlabel(""); ax.set_ylabel("")
    fig.tight_layout(h_pad=2.4, w_pad=2.0)
    save_figure(fig, "08_mouse_annotation_umaps")
    plt.show()
else:
    required = ["leiden", "sample_tissue", "library", "cell_type_detail_final",
                "cell_type_detail_level", "cell_type_final", "cell_type_final_confidence"]
    missing = [c for c in required if c not in adata.obs.columns]
    if missing:
        raise KeyError(f"UMAP metadata fields are missing: {missing}")
    fig, axes = plt.subplots(4, 2, figsize=(18, 28))
    layout = [
        ("leiden", "Leiden cluster", "on data"), ("sample_tissue", "Sample tissue", "right margin"),
        ("atlas_tissue_cell", "Atlas-predicted tissue", "right margin"), ("library", "Library", "right margin"),
        ("cell_type_detail_final", "Refined hierarchical cell type", "right margin"),
        ("cell_type_detail_level", "Detail annotation evidence level", "right margin"),
        ("cell_type_final", "Final broad annotation", "right margin"),
        ("cell_type_final_confidence", "Final broad confidence", "right margin"),
    ]
    for ax, (color, title, legend) in zip(axes.ravel(), layout):
        if color in adata.obs.columns:
            sc.pl.umap(adata, color=color, ax=ax, show=False, title=title, legend_loc=legend, frameon=False)
    for ax in axes.ravel():
        ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout(h_pad=3.0, w_pad=2.0)
    save_figure(fig, "08_census_annotation_umaps")
    plt.show()

# 可选模块 B：DNAme QC 与 RNA–DNAme 联合可视化

**默认关闭**（`ENABLE_DNAME_OPTIONAL = True` 开启；需要 Processor 已生成 Integrated_QC_Information.csv）。DNAme 指标直接来自整合 QC 表，不参与 RNA QC、HVG、PCA、neighbors、Leiden、UMAP 或注释。

In [ ]:
ENABLE_DNAME_OPTIONAL = False

def _require_integrated_frame() -> pd.DataFrame:
    if df_integrated is None:
        raise ValueError("DNAme 模块已开启，但没有可用的 df_integrated（Processor 未做 DNAme 整合）。")
    return df_integrated

DNA_QC_RENAME = {
    "Native_Mapping%": "DNAme_mapping_pct",
    "Duplicate_Pair_Rate%": "DNAme_duplication_pct",
    "PostDedup_Retention%": "DNAme_postdedup_retention_pct",
    "Final_Pair_Yield%": "DNAme_yield_pct",
    "Gini_Index": "DNAme_gini",
    "Sample_CpG_Signals": "DNAme_CpG_signals",
    "Sample_Unique_CpG_Sites": "DNAme_unique_CpGs",
    "Sample_Meth_CpG_Rate%": "DNAme_methylation_pct",
    "Lambda_Unique_CpG_Sites": "DNAme_lambda_unique_CpGs",
    "Lambda_Meth_CpG_Rate%": "DNAme_lambda_methylation_pct",
    "pUC19_Unique_CpG_Sites": "DNAme_pUC19_unique_CpGs",
    "pUC19_Meth_CpG_Rate%": "DNAme_pUC19_methylation_pct",
    "Non_CpG_Methylation%": "DNAme_CpH_methylation_pct",
    "CloneID": "DNAme_clone_id",
    "DNA_Barcode": "DNAme_DNA_barcode",
}

if ENABLE_DNAME_OPTIONAL:
    integrated = _require_integrated_frame()
    missing = [c for c in DNA_QC_RENAME if c not in integrated.columns]
    if missing:
        raise KeyError(f"整合 QC 表缺少 DNAme 列: {missing}")
    dname = integrated[list(DNA_QC_RENAME)].rename(columns=DNA_QC_RENAME).copy()
    dname["DNAme_log10_unique_CpGs"] = np.log10(
        pd.to_numeric(dname["DNAme_unique_CpGs"], errors="coerce").replace(0, np.nan)
    )
    dname.index = integrated["Sample_ID"].astype(str)
    common = adata.obs_names.astype(str).intersection(dname.index)
    if len(common) < adata.n_obs:
        print(f"WARN: {adata.n_obs - len(common)} 个 RNA 细胞没有 DNAme QC 对应行。")
    for column in dname.columns:
        adata.obs[column] = dname.loc[adata.obs_names.astype(str), column].to_numpy()
    print(f"已从整合 QC 表导入 {dname.shape[1]} 个 DNAme 指标到 adata.obs。")
else:
    print("DNAme optional module disabled; set ENABLE_DNAME_OPTIONAL = True.")

In [ ]:
if ENABLE_DNAME_OPTIONAL:
    meth = pd.to_numeric(adata.obs['DNAme_methylation_pct'], errors='coerce').dropna()
    if len(meth) == 0:
        raise ValueError('DNAme 模块已开启，但没有可用 DNAme_methylation_pct。')

    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.8))
    sc.pl.umap(adata, color='cell_type_final', ax=axes[0], show=False,
               size=70, alpha=0.95, legend_loc='right margin',
               title='Final RNA cluster annotation', frameon=False)
    sc.pl.umap(adata, color='DNAme_methylation_pct', ax=axes[1], show=False, cmap='viridis',
               vmin=float(meth.quantile(0.02)), vmax=float(meth.quantile(0.98)),
               size=70, alpha=0.95, title='Global CpG methylation (%)', frameon=False)
    sc.pl.umap(adata, color='DNAme_log10_unique_CpGs', ax=axes[2], show=False, cmap='viridis',
               size=70, alpha=0.95, title='log10 unique CpGs', frameon=False)
    fig.tight_layout()
    save_figure(fig, '09_dname_overlay_umap')
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(8.8, 7.4))
    for ax, color, title in [
        (axes[0, 0], 'DNAme_methylation_pct', 'Global CpG methylation (%)'),
        (axes[0, 1], 'DNAme_log10_unique_CpGs', 'log10 unique CpGs'),
        (axes[1, 0], 'DNAme_mapping_pct', 'DNA mapping (%)'),
        (axes[1, 1], 'DNAme_CpH_methylation_pct', 'CpH methylation (%)'),
    ]:
        vals = pd.to_numeric(adata.obs[color], errors='coerce').dropna()
        if len(vals) == 0:
            ax.set_axis_off(); continue
        sc.pl.umap(adata, color=color, ax=ax, show=False, cmap='viridis',
                   vmin=float(vals.quantile(0.02)), vmax=float(vals.quantile(0.98)),
                   size=70, alpha=0.95, title=title, frameon=False)
    fig.tight_layout()
    save_figure(fig, '09_dname_qc_umaps')
    plt.show()
else:
    print('Skipped RNA-DNAme UMAP overlay.')

In [ ]:
if ENABLE_DNAME_OPTIONAL:
    plot_df = adata.obs.loc[adata.obs['DNAme_methylation_pct'].notna(),
                             ['leiden', 'cell_type', 'library', 'DNAme_methylation_pct']].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 4.4))
    sns.boxplot(data=plot_df, x='cell_type', y='DNAme_methylation_pct', hue='cell_type',
                legend=False, showfliers=False, ax=axes[0])
    axes[0].set_title('DNA methylation by RNA cell type'); axes[0].set_xlabel(''); axes[0].set_ylabel('G')
    axes[0].tick_params(axis='x', rotation=35)
    clean_axes(axes[0], grid_y=True)
    sns.boxplot(data=plot_df, x='library', y='DNAme_methylation_pct', hue='library',
                legend=False, showfliers=False, ax=axes[1])
    axes[1].set_title('DNA methylation by library'); axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=35)
    clean_axes(axes[1], grid_y=True)
    fig.tight_layout()
    save_figure(fig, '09_dname_by_celltype')
    plt.show()
else:
    print('Skipped DNAme boxplots.')

## 结果解释原则

1. 计算全部在 `python -m lemmus.downstream.qc --analyze` 中完成并在 `06_downstream/` 落盘；notebook 只做调参与可视化，重画图不需要重算。
2. human 主注解是 CELLxGENE Census scVI consensus（4 classifier + Cell Ontology 聚合，低一致性保留为 Unresolved）；mouse 主注解是 deterministic cluster-level marker program（支持性检出 + marker gate，低支持 fail closed 到 Uncertain），两者字段合同一致。
3. RNA QC（supported genes/UMI/线粒体）是唯一技术过滤层；DNAme 不参与 RNA 过滤、聚类或注释。
4. Leiden DE marker 用于 sanity review 与小鼠 marker-program 证据；human 主注解不使用单一 marker 规则。
5. seed、阈值、依赖版本和输入哈希写入 `analysis_summary.json` 与 H5AD `uns`，结果可追溯。
6. DNAme 整合表由 Processor 生成；两侧 library 命名不同时按 DNA `00_config/sample_manifest.tsv` 的 `dna_raw_sample -> rna_sample` 桥接。